## Machine learning Piplines 
###  1. Piplines chains together multiple steps so that the output of each step is used as input to the next step
### 2. Pipelines makes it easy to apply the same preprocessing to train and test ! 


### pipeline is use to maintain the whole process of machine learning from scrach 
### from preprocess to output 

In [1]:
### building the ml project without using the piplines

In [2]:
import numpy as np 
import pandas as pd 

from sklearn.model_selection import train_test_split 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder , MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [3]:
df = pd.read_csv("Titanic-Dataset.csv")

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df = df.drop(columns = ['PassengerId','Name','Ticket', 'Cabin'])

In [6]:
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),
                                                 df['Survived'] ,
                                                 test_size=0.2,random_state=42)

In [8]:
X_train.shape

(712, 7)

In [9]:
X_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S
733,2,male,23.0,0,0,13.0,S


In [10]:
y_train.head(2)

331    0
733    0
Name: Survived, dtype: int64

In [11]:
X_train.isnull().sum()

Pclass        0
Sex           0
Age         140
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [12]:
X_test.isnull().sum()

Pclass       0
Sex          0
Age         37
SibSp        0
Parch        0
Fare         0
Embarked     0
dtype: int64

In [41]:
X_train.columns


array(['S', 'C', 'Q', nan], dtype=object)

In [14]:
df['Embarked'].unique()

array(['S', 'C', 'Q', nan], dtype=object)

In [63]:
## Applying imputation to fill the null values 
si_age = SimpleImputer( )     # replacing null values with mean of age 
si_embarked = SimpleImputer(strategy='most_frequent')   # replacing values with most frequent element 

X_train_age = si_age.fit_transform(X_train[['Age']]) 
X_train_embarked = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.transform(X_test[['Age']]) 
X_test_embarked = si_embarked.transform(X_test[['Embarked']])



#### Use fit Transfrom on the training data to fit the data to the encoders and use transform on test data 

In [64]:
## One-Hot encoding on columns Sex , Embarked 
oh_sex = OneHotEncoder(sparse_output=False , handle_unknown='ignore') 
oh_embarked = OneHotEncoder(sparse_output=False,handle_unknown='ignore') 
X_train_sex = oh_sex.fit_transform(X_train[['Sex']])
X_train_embarked = oh_embarked.fit_transform(X_train_embarked)

X_test_sex = oh_sex.transform(X_test[['Sex']])
X_test_embarked = oh_embarked.transform(X_test_embarked)

X_test_embarked[0]

array([1., 0., 0.])

In [65]:
X_train_rem = X_train.drop(columns=['Sex','Embarked','Age'])

In [66]:
X_test_rem = X_test.drop(columns=['Sex','Embarked','Age'])

In [67]:
X_train_rem.shape,X_train_age.shape , X_train_embarked.shape , X_train_sex.shape

((712, 4), (712, 1), (712, 3), (712, 2))

In [68]:
## total 13 columns are there in transformed dataset 

In [69]:
X_train_transformed = np.concatenate((X_train_rem,X_train_age,X_train_embarked,X_train_sex),axis=1)
X_test_transformed = np.concatenate((X_test_rem,X_test_age,X_test_embarked,X_test_sex),axis=1)

In [70]:
X_train.columns

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')

In [71]:
print(X_train_transformed[:1,:] , X_train_transformed[0].shape)   # X_train data 

[[ 1.   0.   0.  28.5 45.5  0.   0.   1.   0.   1. ]] (10,)


In [72]:
clf = DecisionTreeClassifier() 
clf.fit(X_train_transformed,y_train) 

DecisionTreeClassifier()

In [73]:
y_pred = clf.predict(X_test_transformed)

In [74]:
from sklearn.metrics import accuracy_score 

In [75]:
print(f"Accuracy_score : {accuracy_score(y_test,y_pred)} ")

Accuracy_score : 0.7877094972067039 


In [80]:
## storing the model using pickle 
# not storing siimple imputer becouse user not going to miss the values 
# saving  one hot incoders 
import pickle 


In [81]:
pickle.dump(oh_sex,open('models/ohe_sex.pkl','wb'))
pickle.dump(oh_embarked,open('models/ohe_embarked.pkl','wb'))
pickle.dump(clf,open('models/clf.pkl','wb'))

In [82]:
## all the files are gest stored in the folder models 

In [83]:
X_train.columns 

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')

In [84]:
X_train.head(1) 

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S


## Reusing Saved models


In [85]:
ohe_sex = pickle.load(open('models/ohe_sex.pkl','rb'))
ohe_embarked = pickle.load(open('models/ohe_embarked.pkl','rb')) 
model = pickle.load(open('models/clf.pkl','rb'))

ohe_embarked.categories_

[array(['C', 'Q', 'S'], dtype=object)]

In [86]:
sex = ohe_sex.transform([["male"]]) 
embarked = ohe_embarked._transform([["S"]])
print(sex) 
print(embarked)

[[0. 1.]]
(array([[2]]), array([[ True]]))


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [128]:
def input_d() : 
        print("Enter the following details to Predict the Servival... ")
        Pclass = int(input("Enter passenger class : "))
        Sex   = input('Enter sex : ')
        Age   = int(input('Enter you age : '))
        SibSp = int(input('Enter sibsp : '))
        Parch = int(input('Enter Parch : '))
        Fare  = int(input('Enter Fare : '))
        Embarked = input('Enter Embarked : ')  

        sex = ohe_sex.transform([[Sex]]) 
        embarked = ohe_embarked.transform([[Embarked]])

       
        input_data1 = np.concatenate(
           ( np.array([[Pclass]]) ,
            sex ,
            np.array([[Age, SibSp, Parch, Fare]]), 
            embarked), axis=1 
        )
        ## alternate method 
        input_data = [ [Pclass] + list(sex[0]) +[ Age , SibSp , Parch, Fare] + list(embarked[0]) ]
        print(f" Input Data is :{ input_data1}")

        pred = model.predict(input_data1) 

        if pred ==1 : 
            return f'output is survived {pred} ' 
        else : 
            return f'Output is not _survived { pred} '
    
    

In [129]:
X_train.columns

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')

In [130]:
print(input_d())

Enter the following details to Predict the Servival... 


Enter passenger class :  1
Enter sex :  male
Enter you age :  45
Enter sibsp :  0
Enter Parch :  0
Enter Fare :  25
Enter Embarked :  S


 Input Data is :[[ 1.  0.  1. 45.  0.  0. 25.  0.  0.  1.]]
output is survived [1] 


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [57]:
sex = ohe_sex.transform([["male"]]) 
embarked = ohe_embarked._transform([['S']])
print(sex)


[[0. 1.]]
[[2]]


C:\Users\ACER\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


## Directly using pipline 

In [134]:
def input_d() : 
        print("Enter the following details to Predict the Servival... ")
        Pclass = int(input("Enter passenger class : "))
        Sex   = input('Enter sex : ')
        Age   = int(input('Enter you age : '))
        SibSp = int(input('Enter sibsp : '))
        Parch = int(input('Enter Parch : '))
        Fare  = int(input('Enter Fare : '))
        Embarked = input('Enter Embarked : ')  

        pipeline = pickle.load(open('pipeline.pkl','rb'))
    
        pred = pipeline.predict(input_data1) 

        if pred ==1 : 
            return f'output is survived {pred} ' 
        else : 
            return f'Output is not _survived { pred} '
    
    